# HRV Parameter Extraction
This notebook is used for extracting HRV parameters from the peaks extracted in step 3.

To do:
- Calculate R-R intervals (done)
- Plot Frequency domain for HRV values (done)
- Extract SDNN (done)
- Extract pNN50 (done)
- Extract RMSSD (done)
- Extract Frequency domain parameters (done)
- Save these values in the folder associated with the patient (done)

## Initial setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import pyarrow.feather as feather
from tqdm import tqdm

from misc.render import *
from misc.synthetic_noise import *
from misc.noise_filtering import *
from misc.peakdetection import *
from misc.hrvextraction import *

In [ ]:
#%run "Retired/4.0-HRV_Extraction_Function.ipynb"

In [ ]:
#cwd = os.getcwd() # Current Directory
#database_root = os.path.join(cwd,'Databases','Database1') # Local dataset directory

#root = 'data/processed/Database1'
database_root = 'data/processed/Database1' # Local dataset directory

### Function definitions

In [ ]:
import scipy as sp
from scipy.fft import rfft, rfftfreq
from scipy.fft import irfft
from scipy.interpolate import interp1d

def resample(RR_data , freq):
    """ This function aims to resample the signal with a known sampling rate for further frequency domain analysis
    Input: RR_data from the RR_Calculator function and a frequency (freq) for resampling the cubic spline.
    Output: Dataframe called new_RR_data. Columns in new_RR_data are: 
        'x_values' -> the time values of the resampled section
        'R-R Interval' -> The new resampled 'R-R Interval Final' 
    """
    new_RR_data = pd.DataFrame()
    
    # Interpolate with a cubic spline
    x_values = RR_data['x_values'].values
    interpol = interp1d(x_values, RR_data['R-R Interval Final'], kind='cubic')
    
    #Calculate the number of samples we want and resample
    num = int(np.max(x_values)*freq/1000) #Length of the signal * frequency to get total number of samples
    new_RR = sp.signal.resample(interpol(x_values), num) #function is interpol(x_values), num is the number of new samples
    
    #period = 1000/freq #in ms
    
    # Some checks
#     print(period)
#     print(len(new_RR))
#     print(len(np.arange(np.min(x_values), np.max(x_values), period)))

    # Assigning values to new dataFrame
    new_RR_data['R-R Interval Final'] = new_RR
    new_RR_data['x_values'] = np.linspace(np.min(x_values), np.max(x_values), num = num) #Set equally spaced x-axis between beginning and end of original x_values
    new_RR_data['x_values_seconds'] = np.linspace(np.min(x_values)/1000, np.max(x_values)/1000, num = num)
    #np.arange(np.min(x_points), np.max(x_points), period)

    return new_RR_data

def _fft(df,rate):
    """ Function definition for the Fast Fourier Transform and plotting the PSD of the signal.
    Input: Resampled new_RR_data from the resample function and a rate for the sampling rate used in the resample.
    Output: Dataframe called freq_values. Columns in freq_values are: 
        'Frequency' -> the frequencies 
        'Power' -> The power at each frequency 
    """
    # Initialisation
    freq_values = pd.DataFrame()
    time = df['x_values']
    hrv_signal = df['R-R Interval Final'].values

    # Number of samples in normalized_tone
    N = len(hrv_signal)

    # Calculating fourier transforms
    xf = rfftfreq(N, 1 / rate) # Returns the Discrete Fourier Transform sample frequencies
    yf = np.abs(rfft(hrv_signal)) # Computes the one-dimensional discrete Fourier Transform for real input.
    for index,value in enumerate(xf):
        if value < 0.04:
            yf[index] = 0
    
    # Plotting Frequency Domain HRV values
    plt.plot(xf, yf)
    plt.xlabel('Frequency(Hz)')
    plt.ylabel('Power(ms^2)')
    
    #HF - 0.15-0.4, LF - 0.04-0.15
    plt.axvline(x=0.040, color='k')
    plt.axvline(x=0.15, color='r')
    plt.axvline(x=0.40, color='b')
    plt.xlim([0, 0.5])
    #plt.title('PSD Healthy Patient')
    #plt.show()
    
    # Save values in new data_frame
    freq_values['Frequency'] = xf
    freq_values['Power'] = yf
    return freq_values

def _HFpow(freq_values):
    """Finds and returns the absolute power of the High Frequency band (0.15-0.4Hz)
    Input: freq_values - DataFrame of frequency domain HRV values obtained from the fft.
    Output: Absolute power of high frequency band, float
    """
    
    # Select Only the High frequency band
    HF = freq_values.loc[(freq_values['Frequency'] > 0.15) &  (freq_values['Frequency'] < 0.4)]
    HF = HF.reset_index(drop = True)

    # Delta of integration, gap between neigbouring frequency points
    width = (HF.iloc[1]['Frequency'] - HF.iloc[0]['Frequency'])
    
    #'Integrate' the HF band to get absolute power
    HFpow = 0
    for index, row in HF.iterrows():
        HFpow += HF.iloc[index]['Power'] * width
        
    print("New formula being used")
    print('The absolute power of the High Frequency band is', HFpow)
    return HFpow

def _LFpow(freq_values):
    """Finds and returns the absolute power of the Low Frequency band (0.04-0.15Hz)
    Input: freq_values - DataFrame of frequency domain HRV values obtained from the fft.
    Output: Absolute power of high frequency band, float
    """
    
    # Select Only the Low frequency band
    LF = freq_values.loc[(freq_values['Frequency'] > 0.04) &  (freq_values['Frequency'] < 0.15)]
    LF = LF.reset_index(drop = True)
    
    # Delta of integration, gap between neigbouring frequency points
    width = (LF.iloc[1]['Frequency'] - LF.iloc[0]['Frequency'])
    
    #'Integrate' the LF band to get absolute power
    LFpow = 0
    for index, row in LF.iterrows():
        LFpow += LF.iloc[index]['Power'] * width
        
    print('The absolute power of the Low Frequency band is', LFpow)
    return LFpow

def _LFHF(freq_values):
    """Finds and returns the ratio of Low Frequency to High Frequency band powers
    Input: freq_values - DataFrame of frequency domain HRV values obtained from the fft.
    Output: Absolute power of high, low frequency bands and their ratio, all floats
    """
    LF = _LFpow(freq_values)
    HF = _HFpow(freq_values)
    LFHF = LF/HF
    print('The LF/HF is', LFHF)
    return LF, HF, LFHF

def _outlier_removal(hrv):
    """
    The function perfoms a Z-score on the dtaset, removing the data points 2 standard deviations from the mean.
    Inputs
    hrv: dataframe extracted from the RR_calculation function.
    Outputs
    rr: dataframe with the x_values and RR intervals as columns with the outliers removed from the dataframe.
    """
    rr = hrv
    z_scores = sp.stats.zscore(rr)
    abs_z_scores = np.abs(z_scores)
    filtered_entries = (abs_z_scores < 3).all(axis=1)
    index_values = []
    for index,value in enumerate(filtered_entries):
        if value == False:
            index_values.append(index)
    rr = rr.drop(index_values)
    return rr

In [ ]:
def save_parameters(para, path):
    """Save parameters in new json file with snippets"""
    with open(os.path.join(path,'Parameters.json'), "w") as outfile:
        json.dump(para, outfile)
        outfile.close()

In [ ]:
def hrv_analysis(peaks):
    """Function takes a filepath, and extracts R-R interval values + HRV parameters in 
    frequency and time domains using functions from 4.0"""
    
    df = pd.read_feather(peaks + '/PEAKS.ftr')
    plt.figure(figsize=(12,4))
    
    plt.subplot(1,2,1)
    df_HRVt = RR_calculator(df)
    plt.title('HRV values for '  + peaks[-10:] + ' in patient ' + peaks[-23:-18])
    
    plt.subplot(1,2,2)
    df_HRVf = _fft(resample(df_HRVt, 1), 1)
    plt.title('PSD for '  + peaks[-10:] + ' in patient ' + peaks[-23:-18])
    
    para = {}
    para['SDNN'] = SDNN_calculator(df_HRVt) #Calculating the SDNN parameter
    para['RMSSD'] = RMSSD_calculator(df_HRVt) #Calculating the RMSSD parameter
    para['PNN50'] = PNN50_calculator(df_HRVt) #Calculating the PNN50 parameters

    para['LF'], para['HF'], para['LFHF'] = _LFHF(df_HRVf) #Calculating the frequency parameters
    return para

In [ ]:
raw_patients = sorted(os.listdir(database_root))

patients = [patient for patient in raw_patients if not patient.endswith(('.json', 'ini','\r'))]
patients = [patient for patient in raw_patients if os.path.isdir(os.path.join(database_root,patient))]
patients_with_holter = [patient for patient in patients if os.path.isdir(os.path.join(database_root,patient,'holter'))]

# Now we need to check the holter folder contains some snippets ---------------------------------------------
# Helper functions (can be offloaded to another file) -------------------------------------------------------


def is_valid_snippet(snippet):
    # We still need to check the expected files are in the folder
    MSNIP.ftr
    
    return True


def has_snippets(path):
    dirs = [a_dir for a_dir in sorted(os.listdir(path)) if os.path.isdir(os.path.join(path,a_dir))]
    snippet_dirs = [a_dir for a_dir in dirs if a_dir.startswith('Snippet')]

    return len(snippet_dirs)>0


def all_snippets(path):
    dirs = [a_dir for a_dir in sorted(os.listdir(path)) if os.path.isdir(os.path.join(path,a_dir))]
    # now we check they are snippet dirs
    snippet_dirs = [a_dir for a_dir in dirs if a_dir.startswith('Snippet')]
    # valid_snippet_dirs = [a_dir for a_dir in snippet_dirs if is_valid_snippet(os.path.join(path,a_dir))]
    return snippet_dirs


# -----------------------------------------------------------------------------------------------------------
    
patients_with_snippets = [patient for patient in patients_with_holter 
                          if has_snippets(os.path.join(database_root,patient,'holter'))]

print(patients_with_snippets)
# Now we have the patient folders with snippets and we know where they are !
# So the loop will be clean

## Initial Testing on updated Peak detection algorithm
Checking the statistical values calculated for the new snippets.

In [ ]:
for patient in tqdm(patients_with_snippets): #Looping throught the folders Sxxxx with snippets
    patient_path = os.path.join(database_root,patient)
    holter_path = os.path.join(patient_path,'holter')
    all_patient_snippets = all_snippets(holter_path)
    all_snippets_path = [os.path.join(holter_path,snippet) for snippet in all_patient_snippets]

    for path_snippet in all_snippets_path:
        if os.path.exists(os.path.join(path_snippet,'PeakMeta.json')) == True:
            print("In snippet folder " + path_snippet)
            para = hrv_analysis(path_snippet)
            save_parameters(para, path_snippet)
            print('\n\n')
            print('\n\n')

## Initial Testing on Snippet sections selected by time-domain HRV values
### Specific Function definitions

In [ ]:
def check_snippet_valid(path):
    """Checks if the file is valid by checking the error flag in its metadata file"""
    return os.path.exists(path + '/CleanPeakMeta.json')

# Useful but terrible name
# check_snippet_validity? or something like that

In [ ]:
def clean_save_parameters(para, path):
    """Save parameters in new json file with snippets"""
    with open((path + '/CleanParameters.json'), "w") as outfile:
        json.dump(para, outfile)
        outfile.close()

In [ ]:
def visualise_hrv_analysis(peaks):
    """Function takes a filepath, and extracts R-R interval values + HRV parameters in 
    frequency and time domains using functions from 4.0"""
    
    #df = pd.read_feather(peaks + '\\clean_snippet.ftr')
    df = pd.read_feather(peaks + '/clean_snippet.ftr')
    
    df_HRVt = RR_calculator(df)
    df_clean = _outlier_removal(df_HRVt)
    rr = df_clean['R-R Interval Final'].to_list()
    list_end = cos_correction(rr)
    df_clean['R-R Interval Final'] = list_end
    
    plt.figure(figsize=(12,4))
    
    plt.subplot(1,2,1)
    plt.plot(df_HRVt['x_values']/1000,df_HRVt['R-R Interval Final'])
    plt.xlabel('time(s)')
    plt.ylabel('R-R Interval')
    plt.title('HRV values for '  + peaks[-10:] + ' in patient ' + peaks[-23:-18])
    
    plt.subplot(1,2,2)
    df_HRVf = _fft(resample(df_HRVt, 1), 1)
    plt.title('PSD for '  + peaks[-10:] + ' in patient ' + peaks[-23:-18])
    plt.show()
    
    plt.figure(figsize=(12,4))
    
    plt.subplot(1,2,1)
    plt.plot(df_clean['x_values']/1000,df_clean['R-R Interval Final'])
    plt.xlabel('time(s)')
    plt.ylabel('R-R Interval')
    plt.title('HRV values for '  + peaks[-10:] + ' in patient ' + peaks[-23:-18] +':cos corrected')
    
    plt.subplot(1,2,2)
    df_HRVf = _fft(resample(df_clean, 1), 1)
    plt.title('PSD for '  + peaks[-10:] + ' in patient ' + peaks[-23:-18])
    plt.show()
    
    para = {}
    para['SDNN'] =  SDNN_calculator(df_HRVt) #Calculating the SDNN parameter
    para['RMSSD'] = RMSSD_calculator(df_HRVt) #Calculating the RMSSD parameter
    para['PNN50'] = PNN50_calculator(df_HRVt) #Calculating the PNN50 parameters

    para['LF'], para['HF'], para['LFHF'] = _LFHF(df_HRVf) #Calculating the frequency parameters
    return para

In [ ]:
for patient in tqdm(patients_with_snippets): #Looping throught the folders Sxxxx with snippets
    patient_path = os.path.join(database_root,patient)
    holter_path = os.path.join(patient_path,'holter')
    all_patient_snippets = all_snippets(holter_path)
    all_snippets_path = [os.path.join(holter_path,snippet) for snippet in all_patient_snippets]

    for path_snippet in all_snippets_path:
        if os.path.exists(os.path.join(path_snippet,'clean_snippet.ftr')) == True:
            para = visualise_hrv_analysis(path_snippet)
            clean_save_parameters(para, path_snippet)
            print('\n\n')
            